# 01 Legacy pipeline: plays the Databricks notebook

This notebook builds the **legacy side**: three Delta tables in a `legacy_sales` database, just as a Databricks notebook would.

- The rows come from `arf.demo.legacy_rows()`, the same seeded data that `python -m arf demo` uses.
- The Delta files are written to `data/delta/legacy_sales/<table>/` and survive container restarts.

Run it with **Run > Run All Cells**. It starts clean every time.

In [ ]:
import shutil
from datetime import date, datetime
from decimal import Decimal

import lab_common as lab
from arf.demo import LEGACY_TS_FORMAT, legacy_rows

spark = lab.get_spark()
print("Spark", spark.version)

# Fail fast if the BigQuery emulator container is not up (notebook 02 needs it).
print("BigQuery emulator reachable:", list(lab.bq_client().list_datasets()) is not None)

## 1. Start clean

Dropping the database only forgets the table names. Deleting the folder removes the Delta files, so table versions restart at 0.

In [ ]:
spark.sql("DROP DATABASE IF EXISTS legacy_sales CASCADE")
shutil.rmtree(lab.DELTA_ROOT / "legacy_sales", ignore_errors=True)
spark.sql("CREATE DATABASE legacy_sales")

## 2. Create the Delta tables

These are explicit DDL statements, as legacy pipelines usually have. A few details matter later:

- `NOT NULL` on the key columns, so ARF's nullability check has something to compare.
- `customers.created_ts` is **text in IST**, the way an old system often stored it. The migration turns it into a UTC timestamp.
- `unit_price` is `DECIMAL(38,12)`. BigQuery `NUMERIC` keeps only 9 decimals, so ARF should warn about it.

In [ ]:
DDL = {
    "customers": "customer_id BIGINT NOT NULL, customer_name STRING, signup_date DATE, "
                 "created_ts STRING, credit_limit DOUBLE, lifetime_value DECIMAL(18,2)",
    "orders":    "order_id BIGINT NOT NULL, cust_id BIGINT, order_ts TIMESTAMP, quantity INT, "
                 "unit_price DECIMAL(38,12), amount DECIMAL(18,2), discount DOUBLE, status STRING, legacy_flag STRING",
    "inventory": "sku_id BIGINT NOT NULL, sku_code STRING, on_hand INT",
}
for table, columns in DDL.items():
    spark.sql(f"CREATE TABLE legacy_sales.{table} ({columns}) USING DELTA "
              f"LOCATION '{lab.delta_path('legacy_sales', table)}'")
spark.sql("SHOW TABLES IN legacy_sales").show()

## 3. Load the sample data

Python values are converted to the column types: `Decimal` for DECIMAL, `datetime` for TIMESTAMP and `date` for DATE.

In [ ]:
rows = legacy_rows()

data = {
    "customers": [(cid, name, date.fromisoformat(signup), created_ist, credit, Decimal(str(ltv)))
                  for cid, name, signup, created_ist, credit, ltv in rows["customers"]],
    "orders": [(oid, cust, datetime.strptime(ts, LEGACY_TS_FORMAT), qty, Decimal(str(price)),
                Decimal(str(amount)), discount, status, flag)
               for oid, cust, ts, qty, price, amount, discount, status, flag in rows["orders"]],
    "inventory": rows["inventory"],
}

for table, values in data.items():
    schema = spark.table(f"legacy_sales.{table}").schema      # the table's own schema, NOT NULL included
    (spark.createDataFrame(values, schema)
          .write.format("delta").mode("append").saveAsTable(f"legacy_sales.{table}"))
    print(f"legacy_sales.{table}: {spark.table(f'legacy_sales.{table}').count()} rows")

## 4. Look at what you built

`DESCRIBE HISTORY` lists the Delta versions. Version 0 is the empty table from CREATE TABLE, and version 1 is the load.

In [ ]:
spark.sql("SELECT * FROM legacy_sales.orders ORDER BY order_id LIMIT 5").show(truncate=False)
spark.sql("DESCRIBE HISTORY legacy_sales.orders").select("version", "timestamp", "operation").show(truncate=False)

## Optional: the legacy side keeps loading

At the client, legacy pipelines may keep writing while you reconcile. To see what that does to ARF:

1. Run notebook 02 first.
2. Set the flag below to `True` and run this cell. It appends 5 orders as Delta version 2.
3. Run notebook 03. ARF now reports 5 extra rows.
4. Uncomment `as_of_version: 1` in `arf-config/tables/orders.yaml`, then run notebook 03 again. The row-count difference caused by the late rows goes away, because ARF now compares the snapshot that was migrated.

In [ ]:
SIMULATE_LATE_ARRIVALS = False

if SIMULATE_LATE_ARRIVALS:
    late = [(1000 + i, 1, datetime(2026, 2, 1, 12, 0), 1, Decimal("10.000000"), Decimal("10.00"), 0.0, "NEW", "N")
            for i in range(1, 6)]
    (spark.createDataFrame(late, spark.table("legacy_sales.orders").schema)
          .write.format("delta").mode("append").saveAsTable("legacy_sales.orders"))
    spark.sql("DESCRIBE HISTORY legacy_sales.orders").select("version", "operation").show()